In [21]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np

In [22]:
SOURCE_DIR = Path("standardized_extra")

EXPORT_DIR = Path("standardized_exports")

EXPORT_DIR.mkdir(
    exist_ok=True
)

In [23]:
HORIZON_MAP = {
    "agri": 12,
    "energy": 24,

    "m4-hourly": 48,
    "m4-daily": 14,
    "m4-weekly": 13,
    "m4-monthly": 18,
    "m4-quarterly": 8,
    "m4-yearly": 6,

    "traffic": 24,

    "favorita-train": 28,
    "favorita-transaction": 28
}

In [24]:
SIZE_CONFIG = {

    "small": 5,

    "medium": 20,

    "large": None

}

In [25]:
def validate_dataframe(df):

    required_cols = [
        "unique_id",
        "ds",
        "y"
    ]

    missing = [

        c

        for c in required_cols

        if c not in df.columns

    ]

    if missing:

        raise ValueError(
            f"Missing columns: {missing}"
        )

In [26]:
def create_subset(
    df,
    horizon,
    size_name
):

    outputs = []

    SIZE_MULTIPLIERS = {
        "small": 5,
        "medium": 20,
        "large": None
    }

    multiplier = SIZE_MULTIPLIERS[size_name]

    for _, group in df.groupby("unique_id"):

        group = group.sort_values("ds")

        # full dataset
        if multiplier is None:

            outputs.append(group)

            continue

        rows_to_keep = horizon * multiplier

        outputs.append(
            group.tail(
                min(
                    rows_to_keep,
                    len(group)
                )
            )
        )

    return pd.concat(
        outputs,
        ignore_index=True
    )

In [27]:
def enforce_horizon(
    df,
    horizon
):

    outputs = []

    for _, group in df.groupby(
        "unique_id"
    ):

        minimum_rows = (
            horizon * 2
        )

        if len(group) < minimum_rows:

            continue

        outputs.append(group)

    if len(outputs) == 0:

        return pd.DataFrame()

    return pd.concat(
        outputs,
        ignore_index=True
    )

In [28]:
def save_dataset(
    df,
    dataset_name,
    size_name
):

    output_path = (

        EXPORT_DIR

        /

        f"{dataset_name}_{size_name}.csv"

    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {output_path.name}"
    )

In [29]:
def summarize(df):

    return pd.Series({

        "Rows":
            len(df),

        "Series":
            df["unique_id"].nunique(),

        "Start":
            df["ds"].min(),

        "End":
            df["ds"].max()

    })

In [30]:
for file in SOURCE_DIR.glob("*.csv"):

    dataset_name = file.stem

    print(
        f"\nProcessing {dataset_name}"
    )

    if dataset_name not in HORIZON_MAP:

        print(
            "Skipping: no horizon configured"
        ) 
        continue

    horizon = HORIZON_MAP[
        dataset_name
    ]

    df = pd.read_csv(
        file,
        low_memory=False
    )

    validate_dataframe(df)

    print(
        summarize(df)
    )
    for size_name in [
        "small",
        "medium",
        "large"
    ]:

        subset = create_subset(
            df,
            horizon,
            size_name
        )

        subset = enforce_horizon(
            subset,
            horizon
        )

        if len(subset) == 0:

            print(
                f"Skipping {size_name}"
            )

            continue

        save_dataset(
            subset,
            dataset_name,
            size_name
        )


Processing agri
Rows      15490
Series      258
Start      1961
End        2024
dtype: int64
Saved agri_small.csv
Saved agri_medium.csv
Saved agri_large.csv

Processing energy
Rows                 51894720
Series                    370
Start     2011-01-01 00:15:00
End       2015-01-01 00:00:00
dtype: object
Saved energy_small.csv
Saved energy_medium.csv
Saved energy_large.csv

Processing favorita-train
Rows         3369062
Series          5000
Start     2013-01-02
End       2017-08-15
dtype: object
Saved favorita-train_small.csv
Saved favorita-train_medium.csv
Saved favorita-train_large.csv

Processing favorita-transaction
Rows            3351
Series             2
Start     2013-01-02
End       2017-08-15
dtype: object
Saved favorita-transaction_small.csv
Saved favorita-transaction_medium.csv
Saved favorita-transaction_large.csv

Processing m4-daily
Rows      9964658
Series       4227
Start           2
End          9920
dtype: int64
Saved m4-daily_small.csv
Saved m4-daily_medium.csv
